# Etapa 4 — Treinamento de Random Forest e Rede Neural Artificial

Neste notebook, realizamos:
1. **Pipelines Scikit-learn**:
   - `ColumnTransformer` tratando variáveis numéricas (`StandardScaler` para redes neurais, mediana para imputação) e categóricas (`OneHotEncoder`).
2. **Otimização de Hiperparâmetros via Validação Cruzada Agrupada (`GroupKFold`)**:
   - `RandomForestRegressor` (`n_estimators`, `max_depth`, `min_samples_leaf`).
   - `MLPRegressor` (`hidden_layer_sizes`, `alpha`, `learning_rate_init`, `early_stopping`).
3. **Persistência dos modelos finais** via `joblib` e metadados JSON.

In [ ]:
import pandas as pd
from pathlib import Path
from wordgen.modelagem.preparacao import carregar_e_preparar_dados
from wordgen.modelagem.validacao import dividir_treino_teste_agrupado
from wordgen.modelagem.treinamento import treinar_e_otimizar_rf, treinar_e_otimizar_mlp
from wordgen.modelagem.avaliacao import avaliar_modelo, consolidar_tabela_metricas
from wordgen.modelagem.persistencia import salvar_artefato_modelo

DATA_DIR = Path("data") if Path("data").exists() else Path("../../data")
datasets = carregar_e_preparar_dados(DATA_DIR)

cenarios = [
    ("Vocabulário — Exp A", "vocabulario_exp_a", "Vocabulário", "A"),
    ("Vocabulário — Exp B", "vocabulario_exp_b", "Vocabulário", "B"),
    ("TDE II — Exp A", "tde_exp_a", "TDE II", "A"),
    ("TDE II — Exp B", "tde_exp_b", "TDE II", "B"),
]

resultados_modelos = []

for rotulo, chave, inst, exp in cenarios:
    print(f"Treinando modelos para {rotulo}...")
    df = datasets[chave]
    X_train, X_test, y_train, y_test, g_train, g_test = dividir_treino_teste_agrupado(
        df, target_col="ganho", grupo_col="aluno_id", test_size=0.20, random_state=42
    )
    num_cols = ["total_pre"]
    cat_cols = [c for c in X_train.columns if c not in num_cols]

    # RF
    rf_pipe, info_rf = treinar_e_otimizar_rf(X_train, y_train, g_train, num_cols, cat_cols, cv_splits=5, random_state=42)
    diag_rf = avaliar_modelo(rf_pipe, X_test, y_test, "Random Forest", rotulo)
    resultados_modelos.append(diag_rf)

    # MLP
    mlp_pipe, info_mlp = treinar_e_otimizar_mlp(X_train, y_train, g_train, num_cols, cat_cols, cv_splits=5, random_state=42)
    diag_mlp = avaliar_modelo(mlp_pipe, X_test, y_test, "Rede Neural (MLP)", rotulo)
    resultados_modelos.append(diag_mlp)

df_mod_res = consolidar_tabela_metricas(resultados_modelos)
display(df_mod_res)